# Base Station Traffic Spike Prediction

**Dataset:** Shanghai Telecom base station access records, June – November 2014.
Each record is one user session at a base station: user ID, base station coordinates, and session start and end time.

| Section | Content |
|---|---|
| 1 | Environment setup and configuration |
| 2 | Data loading |
| 3 | Data overview |
| 4 | Data cleaning |
| 5 | Clean data export |

## 1. Environment Setup and Configuration

Library imports, display settings, file paths and the validity bounds used in cleaning. All parameters are defined here.

In [1]:
# =============================================================================
# Imports
# =============================================================================
import glob
import json
import os
import platform
import re
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import shapely
from IPython.display import display
from shapely.geometry import shape

# =============================================================================
# Display settings
# =============================================================================
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# =============================================================================
# Paths
# =============================================================================
RAW_DIR       = 'Data/Raw'                                    # 12 raw half-month Excel files (never modified)
DISTRICT_FILE = 'Data/Reference/shanghai_districts.geojson'   # district boundaries (DataV.GeoAtlas)
CLEAN_DIR     = 'Data/Clean'
CLEAN_FILE    = f'{CLEAN_DIR}/sessions_clean.parquet'         # single clean dataset

N_READ_WORKERS = 6    # parallel threads for reading the raw files

# =============================================================================
# Cleaning parameters
# =============================================================================
STUDY_START  = pd.Timestamp('2014-06-01')     # first day of the study period
STUDY_END    = pd.Timestamp('2014-12-01')     # exclusive upper bound
SHANGHAI_LAT = (30.6, 31.9)                   # geographic bounding box of Shanghai
SHANGHAI_LON = (120.8, 122.2)

# =============================================================================
# District assignment
# =============================================================================
DISTRICT_MAX_DIST_KM = 1.0          # max distance to snap a station lying just outside a boundary
DISTRICT_NAMES = {                  # administrative code -> English district name
    '310101': 'Huangpu',   '310104': 'Xuhui',    '310105': 'Changning', '310106': "Jing'an",
    '310107': 'Putuo',     '310109': 'Hongkou',  '310110': 'Yangpu',    '310112': 'Minhang',
    '310113': 'Baoshan',   '310114': 'Jiading',  '310115': 'Pudong',    '310116': 'Jinshan',
    '310117': 'Songjiang', '310118': 'Qingpu',   '310120': 'Fengxian',  '310151': 'Chongming',
}

print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')

Python 3.12.3 | pandas 2.2.2 | numpy 2.2.5


## 2. Data Loading

The raw data comes as 12 half-month Excel files. They are read in parallel with the fast `calamine` Excel engine, kept in chronological order and combined into one DataFrame. The number of records in each file is reported.

In [2]:
def file_sort_key(path):
    # (month, start_day) parsed from a file name such as data_6.1~6.15.xlsx
    month, day = re.search(r'data_(\d+)\.(\d+)~', os.path.basename(path)).groups()
    return int(month), int(day)


def read_file(path):
    # calamine (Rust-based) reader: ~3-4x faster than the default openpyxl engine
    return pd.read_excel(path, engine='calamine')


files = sorted(glob.glob(os.path.join(RAW_DIR, '*.xlsx')), key=file_sort_key)

# Read the files in parallel; map() keeps the chronological order
t0 = time.time()
with ThreadPoolExecutor(max_workers=N_READ_WORKERS) as pool:
    frames = list(pool.map(read_file, files))

file_summary = [{
    'File': os.path.basename(f),
    'Records': len(part),
    'First session': part['start time'].min(),
    'Last session': part['start time'].max(),
} for f, part in zip(files, frames)]

df = pd.concat(frames, ignore_index=True)
del frames

file_summary = pd.DataFrame(file_summary)
file_summary.index = range(1, len(file_summary) + 1)

print(f'Files loaded  : {len(files)}')
print(f'Total records : {len(df):,}')
print(f'Loading time  : {time.time() - t0:.1f} s')
display(file_summary.style.format({'Records': '{:,}'}).set_caption('Table 1. Raw data files'))

Files loaded  : 12
Total records : 6,952,921
Loading time  : 63.4 s


,File,Records,First session,Last session
1,data_6.1~6.15.xlsx,"611,506",2014-05-31 23:00:00,2014-06-15 23:58:47
2,data_6.16~6.30.xlsx,"617,629",2014-06-15 21:01:39,2014-06-30 23:04:23
3,data_7.1~7.15.xlsx,"620,830",2014-06-30 18:41:29,2014-07-15 23:51:50
4,data_7.16~7.31.xlsx,"635,362",2014-07-15 22:26:57,2014-07-31 22:25:27
5,data_8.1~8.15.xlsx,"590,585",2014-07-31 19:20:55,2014-08-15 23:45:07
6,data_8.16~8.31.xlsx,"591,942",2014-08-01 09:36:11,2014-08-31 23:48:03
7,data_9.1~9.15.xlsx,"562,779",2014-08-31 21:39:33,2014-09-15 23:49:00
8,data_9.16~9.30.xlsx,"573,139",2014-08-19 23:00:02,2014-09-30 23:48:39
9,data_10.1~10.15.xlsx,"464,572",2014-09-30 22:57:19,2014-10-15 23:49:40
10,data_10.16~10.31.xlsx,"577,102",2014-10-15 21:59:42,2014-10-31 23:57:49


## 3. Data Overview

Structure of the combined raw data: dimensions, a profile of each column (type, non-null and missing counts, unique values), a preview of the records and descriptive statistics.

In [3]:
# =============================================================================
# Dimensions
# =============================================================================
print(f'Records : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print(f'Memory  : {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB')

# =============================================================================
# Column profile
# =============================================================================
column_profile = pd.DataFrame({
    'Data type': df.dtypes.astype(str),
    'Non-null': df.notna().sum(),
    'Missing': df.isna().sum(),
    'Missing (%)': df.isna().mean() * 100,
    'Unique values': df.nunique(),
})
column_profile.index.name = 'Column'
display(column_profile.style
        .format({'Non-null': '{:,}', 'Missing': '{:,}', 'Missing (%)': '{:.2f}', 'Unique values': '{:,}'})
        .set_caption('Table 2. Column profile of the raw data'))

# =============================================================================
# Record preview
# =============================================================================
display(df.head(10).style.set_caption('Table 3. First 10 records'))
display(df.tail(10).style.set_caption('Table 4. Last 10 records'))

# =============================================================================
# Descriptive statistics (numeric and date-time columns)
# =============================================================================
display(df.describe().T.style
        .format('{:,.4f}', subset=pd.IndexSlice[['latitude', 'longitude'], :])
        .format('{:,.0f}', subset=pd.IndexSlice[['month', 'date'], :])
        .set_caption('Table 5. Descriptive statistics'))

Records : 6,952,921
Columns : 7
Memory  : 855.4 MB


,Data type,Non-null,Missing,Missing (%),Unique values
Column,,,,,
month,int64,"6,952,921",0,0.00,6
date,int64,"6,952,921",0,0.00,31
start time,datetime64[ns],"6,952,921",0,0.00,"5,326,416"
end time,datetime64[ns],"6,952,921",0,0.00,"5,342,942"
latitude,float64,"6,236,620","716,301",10.30,"3,025"
longitude,float64,"6,236,620","716,301",10.30,"3,035"
user id,object,"6,952,921",0,0.00,"9,739"


Column,month,date,start time,end time,latitude,longitude,user id
0,201406,1,2014-06-01 10:22:36,2014-06-01 11:09:52,31.237872,121.470259,edbc54bddf16494a49f39ac057b4185d
1,201406,1,2014-06-01 07:00:37,2014-06-01 08:49:20,31.237872,121.470259,f8206ab58b9bdb070673f7050242e9ee
2,201406,1,2014-06-01 02:49:16,2014-06-01 05:49:12,31.237872,121.470259,f8206ab58b9bdb070673f7050242e9ee
3,201406,1,2014-06-01 08:51:58,2014-06-01 09:20:50,31.237872,121.470259,f8206ab58b9bdb070673f7050242e9ee
4,201406,1,2014-05-31 23:49:10,2014-06-01 02:49:07,31.237872,121.470259,f8206ab58b9bdb070673f7050242e9ee
5,201406,1,2014-05-31 23:39:02,2014-05-31 23:49:05,31.237872,121.470259,f8206ab58b9bdb070673f7050242e9ee
6,201406,1,2014-06-01 17:17:24,2014-06-01 17:18:32,31.237872,121.470259,eb7f2c8f7030376959889a2fae9c3a1d
7,201406,1,2014-06-01 17:17:14,2014-06-01 18:32:29,31.237872,121.470259,eb7f2c8f7030376959889a2fae9c3a1d
8,201406,1,2014-06-01 11:17:02,2014-06-01 11:17:23,31.237872,121.470259,edbc54bddf16494a49f39ac057b4185d
9,201406,1,2014-06-01 10:19:43,2014-06-01 10:20:14,31.237872,121.470259,edbc54bddf16494a49f39ac057b4185d


Column,month,date,start time,end time,latitude,longitude,user id
6952911,201411,30,2014-11-30 05:49:27,2014-11-30 06:33:15,nan,nan,2aa2772a12a1135affd3334961c17811
6952912,201411,30,2014-11-30 07:17:08,2014-11-30 07:39:35,nan,nan,2aa2772a12a1135affd3334961c17811
6952913,201411,30,2014-11-30 20:22:22,2014-11-30 20:46:35,nan,nan,052b6c524b131480b0a4800005cb144e
6952914,201411,30,2014-11-30 20:17:49,2014-11-30 20:17:56,nan,nan,052b6c524b131480b0a4800005cb144e
6952915,201411,30,2014-11-30 17:20:32,2014-11-30 17:20:34,nan,nan,ec22557eed4761b9428bf9fe390292cb
6952916,201411,30,2014-11-30 10:06:19,2014-11-30 10:06:21,nan,nan,ec22557eed4761b9428bf9fe390292cb
6952917,201411,30,2014-11-30 13:02:19,2014-11-30 13:02:21,nan,nan,1566134ec7f832b501df944b13b8631a
6952918,201411,30,2014-11-30 21:02:26,2014-11-30 21:04:44,nan,nan,052b6c524b131480b0a4800005cb144e
6952919,201411,30,2014-11-30 20:50:09,2014-11-30 20:52:30,nan,nan,052b6c524b131480b0a4800005cb144e
6952920,201411,30,2014-11-30 20:58:46,2014-11-30 20:59:50,nan,nan,052b6c524b131480b0a4800005cb144e


,count,mean,min,25%,50%,75%,max,std
Column,,,,,,,,
month,"6,952,921","201,408","201,406","201,407","201,408","201,410","201,411",2
date,"6,952,921",16,1,8,16,23,31,9
start time,6952921,2014-08-28 20:21:48.161975296,2014-05-31 23:00:00,2014-07-13 07:44:44,2014-08-26 17:12:12,2014-10-14 13:59:56,2014-11-30 23:56:40,nan
end time,6952921,2014-08-28 21:00:29.251069184,2014-05-31 23:03:20,2014-07-13 08:23:28,2014-08-26 17:50:17,2014-10-14 14:42:54,2014-11-30 23:57:12,nan
latitude,"6,236,620.0000",31.1977,22.5228,31.1178,31.2043,31.2695,47.3509,0.7442
longitude,"6,236,620.0000",121.3191,102.0040,121.3587,121.4384,121.5139,131.8122,1.3865


## 4. Data Cleaning

Cleaning is applied in five steps. The record count is logged after every step:

| Step | Operation |
|---|---|
| 4.1 | Standardise column names and data types, drop the redundant `date` column |
| 4.2 | Remove duplicate records |
| 4.3 | Remove records with missing values |
| 4.4 | Remove invalid records |
| 4.5 | Assign each record to a Shanghai district |

### 4.1 Standardise Column Names and Data Types

Column names are converted to `snake_case`. Time and coordinate columns are converted to their proper types; any value that cannot be converted becomes missing and is removed in step 4.3. The `date` column (day of month of the logging date) is redundant with `start_time` and is dropped.

In [4]:
# Record count after every cleaning step
cleaning_log = []

def log_step(step, rows_before, rows_after):
    removed = rows_before - rows_after
    cleaning_log.append({'Step': step, 'Records before': rows_before,
                         'Removed': removed, 'Records after': rows_after})
    print(f'{step}')
    print(f'  Records before : {rows_before:>12,}')
    print(f'  Removed        : {removed:>12,}  ({removed / rows_before:.3%})')
    print(f'  Records after  : {rows_after:>12,}')


df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df = df.drop(columns=['date'])

df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
df['end_time']   = pd.to_datetime(df['end_time'], errors='coerce')
df['latitude']   = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude']  = pd.to_numeric(df['longitude'], errors='coerce')

cleaning_log.append({'Step': 'Raw combined data', 'Records before': len(df),
                     'Removed': 0, 'Records after': len(df)})

print(f'Total records: {len(df):,}\n')
display(df.dtypes.astype(str).to_frame('Data type').rename_axis('Column')
          .style.set_caption('Table 6. Standardised columns'))

Total records: 6,952,921



,Data type
Column,
month,int64
start_time,datetime64[ns]
end_time,datetime64[ns]
latitude,float64
longitude,float64
user_id,object


### 4.2 Duplicate Records

A record is a duplicate if it is identical to another record in every column. The first occurrence is kept and the copies are removed.

In [5]:
n_before = len(df)
dup_mask = df.duplicated(keep='first')
n_dup = int(dup_mask.sum())

print(f'Total records     : {n_before:,}')
print(f'Duplicate records : {n_dup:,} ({n_dup / n_before:.3%})\n')

display(df[df.duplicated(keep=False)]
          .sort_values(['user_id', 'start_time'])
          .head(10)
          .style.set_caption('Table 7. Examples of duplicate records (each record and its copy)'))

df = df.loc[~dup_mask].reset_index(drop=True)
del dup_mask

print()
log_step('Remove duplicate records', n_before, len(df))

Total records     : 6,952,921
Duplicate records : 2,055 (0.030%)



,month,start_time,end_time,latitude,longitude,user_id
519151,201406,2014-06-13 15:52:49,2014-06-13 15:53:21,31.246345,121.453710,0019e53be51fb4fd951e1a817d8c54ee
519152,201406,2014-06-13 15:52:49,2014-06-13 15:53:21,31.246345,121.453710,0019e53be51fb4fd951e1a817d8c54ee
1796616,201407,2014-07-14 15:14:35,2014-07-14 15:30:10,31.246345,121.453710,0019e53be51fb4fd951e1a817d8c54ee
1796617,201407,2014-07-14 15:14:35,2014-07-14 15:30:10,31.246345,121.453710,0019e53be51fb4fd951e1a817d8c54ee
2380874,201407,2014-07-29 00:11:24,2014-07-29 01:09:32,31.209904,121.234203,0019e53be51fb4fd951e1a817d8c54ee
2380875,201407,2014-07-29 00:11:24,2014-07-29 01:09:32,31.209904,121.234203,0019e53be51fb4fd951e1a817d8c54ee
337528,201406,2014-06-09 17:17:08,2014-06-09 17:21:00,31.209025,121.616514,0068fcff3197dac425a06891200ea38b
337529,201406,2014-06-09 17:17:08,2014-06-09 17:21:00,31.209025,121.616514,0068fcff3197dac425a06891200ea38b
465507,201406,2014-06-12 03:30:58,2014-06-12 03:31:28,31.209025,121.616514,0068fcff3197dac425a06891200ea38b
465508,201406,2014-06-12 03:30:58,2014-06-12 03:31:28,31.209025,121.616514,0068fcff3197dac425a06891200ea38b



Remove duplicate records
  Records before :    6,952,921
  Removed        :        2,055  (0.030%)
  Records after  :    6,950,866


### 4.3 Missing Values

Missing values are counted per column. Before removal, the missing records are checked for randomness by comparing the missing rate across months, hours of the day and users.

In [6]:
# =============================================================================
# Missing values per column
# =============================================================================
n_before = len(df)
missing = pd.DataFrame({
    'Missing': df.isna().sum(),
    'Missing (%)': df.isna().mean() * 100,
}).rename_axis('Column')
n_rows_missing = int(df.isna().any(axis=1).sum())

print(f'Total records                 : {n_before:,}')
print(f'Records with a missing value  : {n_rows_missing:,} ({n_rows_missing / n_before:.3%})\n')

display(missing.style
        .format({'Missing': '{:,}', 'Missing (%)': '{:.3f}'})
        .set_caption('Table 8. Missing values per column'))

# =============================================================================
# Missingness pattern: is it random?
# =============================================================================
miss = df['latitude'].isna() | df['longitude'].isna()

by_month = pd.DataFrame({
    'Records': df.groupby('month').size(),
    'Missing coordinates': miss.groupby(df['month']).sum(),
}).rename_axis('Month')
by_month['Missing (%)'] = by_month['Missing coordinates'] / by_month['Records'] * 100
display(by_month.style
        .format({'Records': '{:,}', 'Missing coordinates': '{:,}', 'Missing (%)': '{:.2f}'})
        .set_caption('Table 9. Missing coordinates by month'))

by_hour = miss.groupby(df['start_time'].dt.hour).mean() * 100
by_day  = miss.groupby(df['start_time'].dt.normalize()).mean() * 100

per_user = miss.groupby(df['user_id']).agg(missing='sum', rate='mean', records='size')
top_decile = max(1, len(per_user) // 10)
share_missing_top = per_user['missing'].nlargest(top_decile).sum() / miss.sum()
share_records_top = per_user['records'].nlargest(top_decile).sum() / len(df)

print(f'Missing rate by hour of day         : {by_hour.min():.2f}% - {by_hour.max():.2f}%')
print(f'Missing rate by day                 : {by_day.min():.2f}% - {by_day.max():.2f}%')
print(f'Users with >= 1 missing record      : {(per_user["missing"] > 0).sum():,} of {len(per_user):,}')
print(f'Users with all records missing      : {(per_user["rate"] == 1).sum():,}')
print(f'Top 10% of users by missing records : {share_missing_top:.1%} of missing records')
print(f'Top 10% of users by activity        : {share_records_top:.1%} of all records')
del miss, per_user

# =============================================================================
# Removal
# =============================================================================
df = df.dropna().reset_index(drop=True)

print()
log_step('Remove records with missing values', n_before, len(df))

Total records                 : 6,950,866
Records with a missing value  : 715,984 (10.301%)



,Missing,Missing (%)
Column,,
month,0,0.000
start_time,0,0.000
end_time,0,0.000
latitude,"715,984",10.301
longitude,"715,984",10.301
user_id,0,0.000


,Records,Missing coordinates,Missing (%)
Month,,,
201406,"1,228,758","97,641",7.95
201407,"1,255,934","120,859",9.62
201408,"1,182,308","123,489",10.44
201409,"1,135,668","112,085",9.87
201410,"1,041,428","115,146",11.06
201411,"1,106,770","146,764",13.26


Missing rate by hour of day         : 8.47% - 10.99%
Missing rate by day                 : 7.24% - 14.68%
Users with >= 1 missing record      : 6,826 of 9,739
Users with all records missing      : 121
Top 10% of users by missing records : 78.7% of missing records
Top 10% of users by activity        : 49.3% of all records

Remove records with missing values
  Records before :    6,950,866
  Removed        :      715,984  (10.301%)
  Records after  :    6,234,882


**Finding.** Missing coordinates are **not missing completely at random**:

- **Time trend:** the missing rate rises steadily across the study period, from about 8% in June to about 13% in November.
- **User concentration:** a small group of users accounts for most of the missing records, well above their share of overall activity.
- **No daily or hourly clustering:** the rate is stable across hours of the day, and no single day has an outage-like spike.

The station a session belongs to cannot be reliably inferred without coordinates, so these records are removed. As a result, observed base station load is slightly under-counted, more so in later months and for the affected users. This is reported as a limitation of the dataset.

### 4.4 Invalid Records

Each record is checked against the validity rules below. A record can fail more than one rule; any record that fails at least one is removed.

| Rule | Condition for an invalid record |
|---|---|
| Session time order | `end_time` is earlier than `start_time` |
| Study period | `start_time` is outside 1 June – 30 November 2014 |
| Geographic bounds | Coordinates are outside the Shanghai bounding box (lat 30.6–31.9, lon 120.8–122.2) |

In [7]:
n_before = len(df)
rules = {
    'end_time earlier than start_time':
        df['end_time'] < df['start_time'],
    'start_time outside study period':
        (df['start_time'] < STUDY_START) | (df['start_time'] >= STUDY_END),
    'coordinates outside Shanghai':
        ~df['latitude'].between(*SHANGHAI_LAT) | ~df['longitude'].between(*SHANGHAI_LON),
}
invalid_mask = np.logical_or.reduce(list(rules.values()))

invalid_summary = pd.DataFrame({
    'Records': [int(m.sum()) for m in rules.values()],
    'Records (%)': [m.mean() * 100 for m in rules.values()],
}, index=pd.Index(list(rules), name='Rule'))

print(f'Total records   : {n_before:,}')
print(f'Invalid records : {int(invalid_mask.sum()):,} ({invalid_mask.mean():.3%})\n')

display(invalid_summary.style
        .format({'Records': '{:,}', 'Records (%)': '{:.3f}'})
        .set_caption('Table 10. Invalid records by rule'))

df = df.loc[~invalid_mask].reset_index(drop=True)
del rules, invalid_mask

print()
log_step('Remove invalid records', n_before, len(df))

Total records   : 6,234,882
Invalid records : 83,399 (1.338%)



,Records,Records (%)
Rule,,
end_time earlier than start_time,0,0.000
start_time outside study period,"1,024",0.016
coordinates outside Shanghai,"82,398",1.322



Remove invalid records
  Records before :    6,234,882
  Removed        :       83,399  (1.338%)
  Records after  :    6,151,483


**Zero-length sessions.** Sessions where `end_time` equals `start_time` are not invalid: they record a real connection event at the base station, which is the quantity relevant to station load. They are therefore **kept**. Their frequency and concentration are reported below so the decision can be revisited at the modelling stage (they can be identified at any time with `start_time == end_time`).

In [8]:
zero = df['end_time'] == df['start_time']

zero_by_month = pd.DataFrame({
    'Records': df.groupby('month').size(),
    'Zero-length': zero.groupby(df['month']).sum(),
}).rename_axis('Month')
zero_by_month['Zero-length (%)'] = zero_by_month['Zero-length'] / zero_by_month['Records'] * 100

per_user = zero.groupby(df['user_id']).sum()
top_decile = max(1, df['user_id'].nunique() // 10)

print(f'Zero-length sessions                 : {int(zero.sum()):,} ({zero.mean():.3%} of records)')
print(f'Users with >= 1 zero-length session  : {(per_user > 0).sum():,} of {len(per_user):,}')
print(f'Top 10% of users by zero-length count: {per_user.nlargest(top_decile).sum() / zero.sum():.1%} of zero-length sessions\n')

display(zero_by_month.style
        .format({'Records': '{:,}', 'Zero-length': '{:,}', 'Zero-length (%)': '{:.2f}'})
        .set_caption('Table 11. Zero-length sessions by month (retained)'))
del zero, per_user

Zero-length sessions                 : 86,231 (1.402% of records)
Users with >= 1 zero-length session  : 3,002 of 9,616
Top 10% of users by zero-length count: 95.4% of zero-length sessions



,Records,Zero-length,Zero-length (%)
Month,,,
201406,"1,115,922","19,220",1.72
201407,"1,120,598","18,331",1.64
201408,"1,044,913","13,855",1.33
201409,"1,010,192","11,285",1.12
201410,"913,249","12,722",1.39
201411,"946,609","10,818",1.14


### 4.5 District Assignment

Each base station (unique latitude / longitude pair) is assigned to the Shanghai district whose administrative boundary contains it (point-in-polygon test). Boundaries are the 16 current districts from DataV.GeoAtlas (Alibaba Cloud). Under these boundaries, the former Zhabei district, merged in 2015, is part of Jing'an.

A station that falls just outside every boundary, for example on the coastline, is assigned to the nearest district if it is within 1 km. Records at stations further away are removed.

In [9]:
n_before = len(df)

# District polygons
with open(DISTRICT_FILE, encoding='utf-8') as fh:
    geo = json.load(fh)
districts = {DISTRICT_NAMES[str(f['properties']['adcode'])]: shape(f['geometry']) for f in geo['features']}

# Local metric projection (km) around Shanghai, used for boundary distances
KM_PER_DEG_LAT = 110.574
KM_PER_DEG_LON = 111.320 * np.cos(np.radians(31.2))
to_km = lambda geom: shapely.transform(geom, lambda c: c * [KM_PER_DEG_LON, KM_PER_DEG_LAT])

# One lookup per unique base station
stations = df.groupby(['latitude', 'longitude']).size().rename('records').reset_index()
lon, lat = stations['longitude'].to_numpy(), stations['latitude'].to_numpy()
stations['district'] = None
stations['match'] = f'Outside all boundaries (> {DISTRICT_MAX_DIST_KM:g} km)'

# 1) Point-in-polygon
for name, poly in districts.items():
    inside = shapely.contains_xy(poly, lon, lat)
    stations.loc[inside, 'district'] = name
    stations.loc[inside, 'match'] = 'Inside district boundary'

# 2) Nearest district for stations just outside the boundaries
outside = stations['district'].isna().to_numpy()
if outside.any():
    pts_km = to_km(shapely.points(lon[outside], lat[outside]))
    dist_km = np.column_stack([shapely.distance(to_km(p), pts_km) for p in districts.values()])
    nearest = np.array(list(districts))[dist_km.argmin(axis=1)]
    snap = dist_km.min(axis=1) <= DISTRICT_MAX_DIST_KM
    idx = np.flatnonzero(outside)[snap]
    stations.loc[idx, 'district'] = nearest[snap]
    stations.loc[idx, 'match'] = f'Nearest district (<= {DISTRICT_MAX_DIST_KM:g} km)'

match_summary = (stations.groupby('match')
                   .agg(Stations=('records', 'size'), Records=('records', 'sum'))
                   .rename_axis('Assignment'))
match_summary['Records (%)'] = match_summary['Records'] / n_before * 100
display(match_summary.style
        .format({'Stations': '{:,}', 'Records': '{:,}', 'Records (%)': '{:.3f}'})
        .set_caption('Table 12. District assignment of base stations'))

# Attach the district to every record and drop records without one
df = df.merge(stations[['latitude', 'longitude', 'district']], on=['latitude', 'longitude'], how='left')
df = df.dropna(subset=['district']).reset_index(drop=True)
df = df[['month', 'start_time', 'end_time', 'latitude', 'longitude', 'district', 'user_id']]

district_summary = pd.DataFrame({
    'Stations': stations.dropna(subset=['district']).groupby('district').size(),
    'Records': df.groupby('district').size(),
}).rename_axis('District')
district_summary['Records (%)'] = district_summary['Records'] / len(df) * 100
district_summary = district_summary.sort_values('Records', ascending=False)
display(district_summary.style
        .format({'Stations': '{:,}', 'Records': '{:,}', 'Records (%)': '{:.2f}'})
        .set_caption('Table 13. Base stations and records per district'))

print()
log_step('Remove records outside Shanghai districts', n_before, len(df))

,Stations,Records,Records (%)
Assignment,,,
Inside district boundary,"2,997","6,145,183",99.898
Nearest district (<= 1 km),9,"5,772",0.094
Outside all boundaries (> 1 km),3,528,0.009


,Stations,Records,Records (%)
District,,,
Pudong,635,"1,282,331",20.85
Minhang,234,"861,805",14.01
Baoshan,192,"546,787",8.89
Songjiang,165,"464,754",7.56
Xuhui,203,"408,277",6.64
Jiading,161,"364,104",5.92
Yangpu,167,"340,431",5.53
Putuo,168,"299,515",4.87
Jing'an,208,"282,875",4.60



Remove records outside Shanghai districts
  Records before :    6,151,483
  Removed        :          528  (0.009%)
  Records after  :    6,150,955


### 4.6 Cleaning Summary

The record count at each stage of cleaning, followed by a final check that the clean data has no duplicates and no missing values.

In [10]:
cleaning_summary = pd.DataFrame(cleaning_log)
cleaning_summary['Removed (%)'] = cleaning_summary['Removed'] / cleaning_summary['Records before'] * 100
cleaning_summary.index = range(len(cleaning_summary))

total_removed = cleaning_summary['Records before'].iloc[0] - cleaning_summary['Records after'].iloc[-1]

display(cleaning_summary.style
        .format({'Records before': '{:,}', 'Removed': '{:,}', 'Records after': '{:,}', 'Removed (%)': '{:.3f}'})
        .set_caption('Table 14. Data cleaning summary'))

print(f'Raw records        : {cleaning_summary["Records before"].iloc[0]:>12,}')
print(f'Total removed      : {total_removed:>12,}  ({total_removed / cleaning_summary["Records before"].iloc[0]:.3%})')
print(f'Clean records      : {len(df):>12,}')
print(f'Duplicates left    : {int(df.duplicated().sum()):>12,}')
print(f'Missing values left: {int(df.isna().sum().sum()):>12,}')
print(f'Unique users       : {df["user_id"].nunique():>12,}')
print(f'Unique stations    : {df.groupby(["latitude", "longitude"]).ngroups:>12,}')
print(f'Period             : {df["start_time"].min():%Y-%m-%d} to {df["start_time"].max():%Y-%m-%d}')

,Step,Records before,Removed,Records after,Removed (%)
0,Raw combined data,"6,952,921",0,"6,952,921",0.000
1,Remove duplicate records,"6,952,921","2,055","6,950,866",0.030
2,Remove records with missing values,"6,950,866","715,984","6,234,882",10.301
3,Remove invalid records,"6,234,882","83,399","6,151,483",1.338
4,Remove records outside Shanghai districts,"6,151,483",528,"6,150,955",0.009


Raw records        :    6,952,921
Total removed      :      801,966  (11.534%)
Clean records      :    6,150,955
Duplicates left    :            0
Missing values left:            0
Unique users       :        9,616
Unique stations    :        3,006
Period             : 2014-06-01 to 2014-11-30


## 5. Clean Data Export

All clean records are saved in chronological order to a single Parquet file. Parquet is a compressed columnar format that keeps the column data types (for example date-times) and loads in seconds. The file is read back and compared with the in-memory data to confirm the export is complete and unchanged.

In [11]:
df = df.sort_values('start_time').reset_index(drop=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

t0 = time.time()
df.to_parquet(CLEAN_FILE, index=False)
write_time = time.time() - t0

# Verify: read the file back and compare with the in-memory data
t0 = time.time()
check = pd.read_parquet(CLEAN_FILE)
read_time = time.time() - t0
assert check.equals(df), 'Exported file does not match the clean data'

export_summary = pd.DataFrame([{
    'File': os.path.basename(CLEAN_FILE),
    'Records': len(check),
    'Columns': check.shape[1],
    'Size (MB)': os.path.getsize(CLEAN_FILE) / 1024**2,
    'First session': check['start_time'].min(),
    'Last session': check['start_time'].max(),
}]).set_index('File')
del check

print(f'Saved to     : {CLEAN_FILE}')
print(f'Write time   : {write_time:.1f} s')
print(f'Read time    : {read_time:.1f} s')
print('Verification : file matches the clean data\n')
display(export_summary.style
        .format({'Records': '{:,}', 'Size (MB)': '{:.1f}'})
        .set_caption('Table 15. Exported clean data file'))

Saved to     : Data/Clean/sessions_clean.parquet
Write time   : 2.1 s
Read time    : 0.7 s
Verification : file matches the clean data



,Records,Columns,Size (MB),First session,Last session
File,,,,,
sessions_clean.parquet,"6,150,955",7,102.4,2014-06-01 00:00:05,2014-11-30 23:56:40
